In [ ]:
!pip install pennylane

**<h1>Angle feature mapping**

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

# number of qubits / features
n_qubits = 10

# device
dev = qml.device("default.qubit", wires=n_qubits)

# ============================================================
# DENSE ANGLE FEATURE MAP
# ============================================================
def feature_map(x):

    for i in range(n_qubits):
        val = x[i] if i < len(x) else 0

        qml.RX(val, wires=i)
        qml.RY(2 * val, wires=i)
        qml.RZ(0.5 * val, wires=i)

    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])


# ============================================================
# QNODE
# ============================================================
@qml.qnode(dev)
def circuit(x):

    feature_map(x)

    return qml.state()


# sample input (10 features)
x = np.random.rand(10)

# ============================================================
# VISUALIZATION
# ============================================================

fig, ax = qml.draw_mpl(circuit)(x)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('/content/survey lung cancer.csv')

In [ ]:
dfen2 = df.copy()
dfen2['GENDER'] = dfen2['GENDER'].map({'F':0,'M':1})
dfen2['LUNG_CANCER'] = dfen2['LUNG_CANCER'].map({'NO':0,'YES':1})

In [ ]:
# ------------------------------------------
#  Install imblearn once (if not installed)
# ------------------------------------------
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# ------------------------------------------
# 1️⃣  Verify dataset & identify target column
# ------------------------------------------
# You already have dfs
print("Original shape:", dfen2.shape)
print("Original class distribution:")
print(dfen2['LUNG_CANCER'].value_counts())

# ------------------------------------------
# 2️⃣  Separate features (X) and target (y)
# ------------------------------------------
X = dfen2.drop(columns=['LUNG_CANCER']).values
y = dfen2['LUNG_CANCER'].values

# ------------------------------------------
# 3️⃣  Apply SMOTE only on the minority class
# ------------------------------------------
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ------------------------------------------
# 4️⃣  Rebuild into a balanced DataFrame
# ------------------------------------------
columns = dfen2.drop(columns=['LUNG_CANCER']).columns
dfs_smotes = pd.DataFrame(X_res, columns=columns)
dfs_smotes['LUNG_CANCER'] = y_res


# ------------------------------------------
# 5️⃣  Check the new class balance
# ------------------------------------------
print("\nAfter SMOTE:")
print(dfs_smotes['LUNG_CANCER'].value_counts())
print("New shape:", dfs_smotes.shape)

In [ ]:
dff=dfs_smotes[:]
dff

In [ ]:
dff.info()

**<h1>Qsvm**

In [ ]:
import numpy as np
import pandas as pd
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    cohen_kappa_score
)
from sklearn.svm import SVC

import pennylane as qml
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
TEST_SIZE = 0.8
NUM_REPEATS = 65
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "pennylane_qsvm_results.csv"
TARGET = "LUNG_CANCER"

FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]

# ============================================================
# DATA
# ============================================================
try:
    dfs_smotes
except NameError:
    print("Creating synthetic dataset...")
    np.random.seed(42)
    n_samples=200
    n_features=15

    X_pos=np.random.randn(n_samples//2,n_features)+1
    X_neg=np.random.randn(n_samples//2,n_features)-1

    X=np.vstack([X_pos,X_neg])
    y=np.array([1]*(n_samples//2)+[0]*(n_samples//2))

    cols=[f"feature_{i}" for i in range(n_features)]
    dfs_smotes=pd.DataFrame(X,columns=cols)
    dfs_smotes[TARGET]=y

X_raw = dfs_smotes.iloc[:,FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true,y_prob,thr=0.5):

    y_pred=(y_prob>=thr).astype(int)

    tn,fp,fn,tp=confusion_matrix(y_true,y_pred).ravel()

    return {
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,zero_division=0),
        "Recall":recall_score(y_true,y_pred,zero_division=0),
        "F1":f1_score(y_true,y_pred,zero_division=0),
        "ROC-AUC":roc_auc_score(y_true,y_prob),
        "PR-AUC":average_precision_score(y_true,y_prob),
        "Sensitivity":tp/(tp+fn) if(tp+fn)>0 else 0,
        "Specificity":tn/(tn+fp) if(tn+fp)>0 else 0,
        "Kappa":cohen_kappa_score(y_true,y_pred)
    }

# ============================================================
# QUANTUM DEVICE
# ============================================================
n_qubits=len(FIXED_FEATURES_IDX)

dev=qml.device("default.qubit",wires=n_qubits)

# ============================================================
# DENSE ANGLE FEATURE MAP
# ============================================================
def feature_map(x):

    for i in range(n_qubits):
        val=x[i] if i<len(x) else 0

        qml.RX(val,wires=i)
        qml.RY(2*val,wires=i)
        qml.RZ(0.5*val,wires=i)

    for i in range(n_qubits-1):
        qml.CNOT(wires=[i,i+1])

# ============================================================
# QNODE
# ============================================================
@qml.qnode(dev)
def circuit(x):

    feature_map(x)

    return qml.state()

# ============================================================
# KERNEL FUNCTION
# ============================================================
def quantum_kernel(X1,X2):

    states1=[circuit(x) for x in X1]
    states2=[circuit(x) for x in X2]

    K=np.zeros((len(X1),len(X2)))

    for i in range(len(X1)):
        for j in range(len(X2)):

            K[i,j]=np.abs(np.vdot(states1[i],states2[j]))**2

    return K

# ============================================================
# MAIN LOOP
# ============================================================
summary=[]

for split in range(1,NUM_REPEATS+1):

    X_train,X_test,y_train,y_test=train_test_split(
        X_raw,y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()

    X_train=sc.fit_transform(imp.fit_transform(X_train))
    X_test=sc.transform(imp.transform(X_test))

    start=time.time()

    K_train=quantum_kernel(X_train,X_train)
    K_test=quantum_kernel(X_test,X_train)

    clf=SVC(kernel="precomputed",probability=True)

    clf.fit(K_train,y_train)

    y_prob=clf.predict_proba(K_test)[:,1]

    runtime=time.time()-start

    metrics=compute_metrics(y_test,y_prob)

    print(

        f"[{progress:6.2f}%] "
        f"_QSVM | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row={
        "Split":split,
        "Accuracy":metrics["Accuracy"],
        "ROC_AUC":metrics["ROC-AUC"],
        "F1":metrics["F1"],
        "Precision":metrics["Precision"],
        "Sensitivity":metrics["Sensitivity"],
        "Specificity":metrics["Specificity"],
        "Kappa":metrics["Kappa"],
        "Runtime_sec":runtime
    }

    summary.append(row)

    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH,index=False)

print("\nDONE")

**<h1>Qknn**

In [ ]:
import numpy as np
import pandas as pd
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    cohen_kappa_score
)

import pennylane as qml
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
TEST_SIZE = 0.8
NUM_REPEATS = 20
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "pennylane_qknn_results.csv"
TARGET = "LUNG_CANCER"

FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]

# QKNN parameter
K_NEIGHBORS = 3

# ============================================================
# DATA
# ============================================================
try:
    dfs_smotes
except NameError:
    print("Creating synthetic dataset...")
    np.random.seed(42)
    n_samples=200
    n_features=15

    X_pos=np.random.randn(n_samples//2,n_features)+1
    X_neg=np.random.randn(n_samples//2,n_features)-1

    X=np.vstack([X_pos,X_neg])
    y=np.array([1]*(n_samples//2)+[0]*(n_samples//2))

    cols=[f"feature_{i}" for i in range(n_features)]
    dfs_smotes=pd.DataFrame(X,columns=cols)
    dfs_smotes[TARGET]=y

X_raw = dfs_smotes.iloc[:,FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true,y_prob,thr=0.5):

    y_pred=(y_prob>=thr).astype(int)

    tn,fp,fn,tp=confusion_matrix(y_true,y_pred).ravel()

    return {
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,zero_division=0),
        "Recall":recall_score(y_true,y_pred,zero_division=0),
        "F1":f1_score(y_true,y_pred,zero_division=0),
        "ROC-AUC":roc_auc_score(y_true,y_prob),
        "PR-AUC":average_precision_score(y_true,y_prob),
        "Sensitivity":tp/(tp+fn) if(tp+fn)>0 else 0,
        "Specificity":tn/(tn+fp) if(tn+fp)>0 else 0,
        "Kappa":cohen_kappa_score(y_true,y_pred)
    }

# ============================================================
# QUANTUM DEVICE
# ============================================================
n_qubits=len(FIXED_FEATURES_IDX)

dev=qml.device("default.qubit",wires=n_qubits)

# ============================================================
# DENSE ANGLE FEATURE MAP
# ============================================================
def feature_map(x):

    for i in range(n_qubits):
        val=x[i] if i<len(x) else 0

        qml.RX(val,wires=i)
        qml.RY(2*val,wires=i)
        qml.RZ(0.5*val,wires=i)

    for i in range(n_qubits-1):
        qml.CNOT(wires=[i,i+1])

# ============================================================
# QNODE
# ============================================================
@qml.qnode(dev)
def circuit(x):

    feature_map(x)

    return qml.state()

# ============================================================
# KERNEL FUNCTION
# ============================================================
def quantum_kernel(X1,X2):

    states1=[circuit(x) for x in X1]
    states2=[circuit(x) for x in X2]

    K=np.zeros((len(X1),len(X2)))

    for i in range(len(X1)):
        for j in range(len(X2)):

            K[i,j]=np.abs(np.vdot(states1[i],states2[j]))**2

    return K

# ============================================================
# QKNN PREDICT
# ============================================================
def qknn_predict(K_test, y_train, k=3):

    probs=[]

    for i in range(K_test.shape[0]):

        sims=K_test[i]

        idx=np.argsort(sims)[-k:]

        labels=y_train[idx]

        prob=np.mean(labels)

        probs.append(prob)

    return np.array(probs)

# ============================================================
# MAIN LOOP
# ============================================================
summary=[]

total_jobs=NUM_REPEATS
completed_jobs=0

for split in range(1,NUM_REPEATS+1):

    X_train,X_test,y_train,y_test=train_test_split(
        X_raw,y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()

    X_train=sc.fit_transform(imp.fit_transform(X_train))
    X_test=sc.transform(imp.transform(X_test))

    start=time.time()

    K_train=quantum_kernel(X_train,X_train)
    K_test=quantum_kernel(X_test,X_train)

    y_prob=qknn_predict(K_test,y_train,K_NEIGHBORS)

    runtime=time.time()-start

    metrics=compute_metrics(y_test,y_prob)

    completed_jobs+=1
    progress=(completed_jobs/total_jobs)*100

    print(

        f"[{progress:6.2f}%] "
        f"PennyLane_QKNN | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row={
        "Split":split,
        "Accuracy":metrics["Accuracy"],
        "ROC_AUC":metrics["ROC-AUC"],
        "F1":metrics["F1"],
        "Precision":metrics["Precision"],
        "Sensitivity":metrics["Sensitivity"],
        "Specificity":metrics["Specificity"],
        "Kappa":metrics["Kappa"],
        "Runtime_sec":runtime
    }

    summary.append(row)

    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH,index=False)

print("\nDONE")

**<h1>Qboost**

In [ ]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    cohen_kappa_score
)

import pennylane as qml
from pennylane import numpy as pnp

# ================================
# USER CONTROLS
# ================================
TEST_SIZE = 0.8
NUM_REPEATS = 65     # Increase later
RANDOM_SEED_BASE = 42
TARGET = "LUNG_CANCER"
FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]

# ================================
# LOAD DATASET
# ================================
# Make sure dfs_smotes exists
try:
    dfs_smotes
except NameError:
    raise ValueError("Dataset 'dfs_smotes' not found. Please load it first.")

# Extract features and target
X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)

# ================================
# METRICS
# ================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# ================================
# QUANTUM DEVICE
# ================================
n_qubits = len(FIXED_FEATURES_IDX)
dev = qml.device("default.qubit", wires=n_qubits)

# ================================
# FEATURE MAP
# ================================
def feature_map(x):
    for i in range(n_qubits):
        qml.RX(x[i], wires=i)
        qml.RY(2*x[i], wires=i)
        qml.RZ(0.5*x[i], wires=i)
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])

# ================================
# QNODE WEAK LEARNER
# ================================
@qml.qnode(dev, interface="autograd")
def qnode(x, weights):
    feature_map(x)
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# ================================
# QUANTUM WEAK LEARNER CLASS
# ================================
class QuantumWeakLearner:
    def __init__(self, n_qubits, n_layers=1, lr=0.1, max_iter=20):
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.lr = lr
        self.max_iter = max_iter
        self.weights = pnp.random.randn(n_layers, n_qubits)
        self.alpha = 0

    def fit(self, X, y, sample_weights):
        opt = qml.AdamOptimizer(self.lr)
        y_trans = 2*y - 1
        for _ in range(self.max_iter):
            for i, x in enumerate(X):
                self.weights = opt.step(lambda w: -sample_weights[i] * y_trans[i] * pnp.mean(pnp.array(qnode(x, w))), self.weights)
        y_pred = np.array([1 if pnp.mean(pnp.array(qnode(x, self.weights)))>0 else 0 for x in X])
        err = np.sum(sample_weights * (y_pred != y)) / np.sum(sample_weights)
        self.alpha = 0.5 * np.log((1 - err) / max(err, 1e-10))
        return self

    def predict(self, X):
        return np.array([1 if pnp.mean(pnp.array(qnode(x, self.weights)))>0 else 0 for x in X])

    def predict_score(self, X):
        return np.array([pnp.mean(pnp.array(qnode(x, self.weights))) * self.alpha for x in X])

# ================================
# QBOOST IMPLEMENTATION
# ================================
class QBoost:
    def __init__(self, n_estimators=5, n_layers=1, lr=0.1, max_iter=20):
        self.n_estimators = n_estimators
        self.n_layers = n_layers
        self.lr = lr
        self.max_iter = max_iter
        self.learners = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        sample_weights = np.ones(n_samples) / n_samples
        self.learners = []

        for _ in range(self.n_estimators):
            learner = QuantumWeakLearner(n_qubits=n_qubits, n_layers=self.n_layers, lr=self.lr, max_iter=self.max_iter)
            learner.fit(X, y, sample_weights)
            y_pred = learner.predict(X)
            err = np.sum(sample_weights * (y_pred != y)) / np.sum(sample_weights)
            if err > 0.5:
                continue
            sample_weights = sample_weights * np.exp(-learner.alpha * (2*y - 1) * (2*y_pred - 1))
            sample_weights /= np.sum(sample_weights)
            self.learners.append(learner)
        return self

    def predict_proba(self, X):
        score = np.zeros(X.shape[0])
        for learner in self.learners:
            score += learner.predict_score(X)
        prob = 1/(1 + np.exp(-score))
        return np.vstack([1-prob, prob]).T

    def predict(self, X):
        return (self.predict_proba(X)[:,1]>=0.5).astype(int)

# ================================
# TRAIN AND EVALUATE
# ================================
summary = []

for split in range(1, NUM_REPEATS+1):
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    start = time.time()
    clf = QBoost(n_estimators=3, n_layers=1, lr=0.1, max_iter=10)
    clf.fit(X_train, y_train)
    y_prob = clf.predict_proba(X_test)[:,1]
    runtime = time.time() - start

    metrics = compute_metrics(y_test, y_prob)

    print(
        f"[{split:2d}/{NUM_REPEATS:2d}] "
        f"QBoost | Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sens={metrics['Sensitivity']:.4f} | "
        f"Spec={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    summary.append({
        "Split": split,
        **metrics,
        "Runtime_sec": runtime
    })

df_summary = pd.DataFrame(summary)
print(df_summary)